# SBML validation

Validates every model in `02_final_annotated/` (the output of `001_standardization.ipynb`, and
the model set the downstream analysis notebooks load) using libSBML, the same way
`GEMcompare/Pipeline/1_Preparation/1.2_Validation.py` did for the thesis: `checkInternalConsistency()`
and `checkL3v2Compatibility()` on the raw `SBMLDocument`.

Unlike that script, a model is only reported as **FAIL** if libSBML raises a **Fatal**-severity
issue (one that means the document cannot be trusted/parsed correctly). Non-fatal `Error`- and
`Warning`-severity issues (e.g. missing optional annotations, unit inconsistencies) are logged
for visibility but don't flip the verdict to FAIL - these are common in third-party GEMs and
don't indicate a broken model.

In [1]:
import glob
import os

import libsbml

final_dir = os.path.join(os.getcwd(), "..", "data", "models", "02_final_annotated")
model_paths = sorted(glob.glob(os.path.join(final_dir, "*.xml")))
print(f"[INFO] Found {len(model_paths)} model(s) in {final_dir}")

[INFO] Found 10 model(s) in C:\Users\felip\Desktop\python\Bacillus_GEMs_Paper copy\analysis\model_preparation\..\data\models\02_final_annotated


## Validation checks

For each model: read the SBML file, then run libSBML's internal-consistency and SBML L3v2
compatibility checks against the document. Every reported issue is classified by severity
(`Fatal`, `Error`, `Warning`, `Info`) via `SBMLError.isFatal()` / `getSeverityAsString()`.

In [2]:
def run_validation_check(document, check_function, check_name):
    """Run one libSBML validation check and classify every reported issue by severity."""
    print(f"[INFO] Running {check_name}")

    issue_count = getattr(document, check_function)()
    issues = []
    fatal_count = 0

    for i in range(issue_count):
        error = document.getError(i)
        severity = error.getSeverityAsString()
        if error.isFatal():
            fatal_count += 1
        issues.append(f"[{severity}] {error.getMessage()}")

    if issue_count == 0:
        print(f"[OK] {check_name}: no issues")
    else:
        print(f"[INFO] {check_name}: {issue_count} issue(s) reported, {fatal_count} fatal")
        for issue in issues:
            print(f"  {issue}")

    return {"check": check_name, "issues": issues, "fatal_count": fatal_count}


def validate_model(model_path):
    """Load an SBML model and run the internal-consistency and L3v2-compatibility checks on it."""
    reader = libsbml.SBMLReader()
    document = reader.readSBML(model_path)

    # Fatal read errors (malformed XML, etc.) - distinct from the validation checks below,
    # which need a document libSBML could parse in the first place.
    read_fatal = sum(
        1 for i in range(document.getNumErrors()) if document.getError(i).isFatal()
    )
    if read_fatal:
        print(f"[ERROR] {read_fatal} fatal error(s) reading the SBML document:")
        document.printErrors()

    checks = [
        ("checkInternalConsistency", "Internal consistency"),
        ("checkL3v2Compatibility", "SBML L3v2 compatibility"),
    ]
    results = [run_validation_check(document, func, desc) for func, desc in checks]

    total_fatal = read_fatal + sum(r["fatal_count"] for r in results)
    return results, total_fatal

## Run validation over all models

In [3]:
summary = []

for model_path in model_paths:
    name = os.path.splitext(os.path.basename(model_path))[0]
    print(f"\n[INFO] Validating {name} ({model_path})")

    results, total_fatal = validate_model(model_path)
    status = "PASS" if total_fatal == 0 else f"FAIL ({total_fatal} fatal issue(s))"
    print(f"[{'OK' if total_fatal == 0 else 'ERROR'}] {name}: {status}")

    summary.append(
        {
            "model": name,
            "status": status,
            "total_issues": sum(len(r["issues"]) for r in results),
            "fatal_issues": total_fatal,
        }
    )


[INFO] Validating Submodel (C:\Users\felip\Desktop\python\Bacillus_GEMs_Paper copy\analysis\model_preparation\..\data\models\02_final_annotated\Submodel.xml)


[INFO] Running Internal consistency


[OK] Internal consistency: no issues
[INFO] Running SBML L3v2 compatibility
[OK] SBML L3v2 compatibility: no issues
[OK] Submodel: PASS

[INFO] Validating ecBSU1 (C:\Users\felip\Desktop\python\Bacillus_GEMs_Paper copy\analysis\model_preparation\..\data\models\02_final_annotated\ecBSU1.xml)
[INFO] Running Internal consistency


[INFO] Internal consistency: 25 issue(s) reported, 0 fatal


  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C13H22NO11PR2(C5H8O6PR)n' has incorrect syntax.

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C13H22NO11PR2(C5H8O6PR)n' has incorrect syntax.

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C16H29N2O11PR2(C5H8

...

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C20H28N6O13PR(C5H8O6PR)n' has incorrect syntax.

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C14H22NO13PR2(C5H8O6PR)n' has incorrect syntax.

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C13H22NO12PR2(C5H8O6

[INFO] Running Internal consistency


[INFO] Internal consistency: 55 issue(s) reported, 0 fatal
  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula 'C25H30N8O10(C5H7NO3)n-2' has incorrect syntax.

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter. The chemicalFormula '(C5H8O4)n' has incorrect syntax.

  [Error] The value of attribute 'fbc:chemicalFormula' on the SBML <species> object must be set to a string consisting only of atomic names or user defined compounds and their occurrence.
Reference: L3V1 Fbc V3 Section 3.4
 Encountered '(' when expecting a capital letter

[INFO] Running Internal consistency
[OK] Internal consistency: no issues
[INFO] Running SBML L3v2 compatibility
[OK] SBML L3v2 compatibility: no issues


[OK] iYO844: PASS


## Summary

In [4]:
import pandas as pd

summary_df = pd.DataFrame(summary)
display(summary_df)

failed = summary_df[summary_df["fatal_issues"] > 0]
if failed.empty:
    print(f"[INFO] All {len(summary_df)} model(s) PASS (no fatal errors; warnings/non-fatal errors ignored)")
else:
    print(f"[WARNING] {len(failed)} model(s) FAILED with fatal errors: {failed['model'].tolist()}")

,model,status,total_issues,fatal_issues
0,Submodel,PASS,0,0
1,ecBSU1,PASS,25,0
2,eciYO844,PASS,0,0
3,iBB1018,PASS,0,0
4,iBsu1103,PASS,0,0
5,iBsu1103v2,PASS,0,0
6,iBsu1147,PASS,25,0
7,iBsu1147R,PASS,25,0
8,iBsu1209,PASS,55,0
9,iYO844,PASS,0,0


[INFO] All 10 model(s) PASS (no fatal errors; warnings/non-fatal errors ignored)
